In [ ]:
import sqlite3
import pandas as pd

db = "../database/futures.db"

conn = sqlite3.connect(db)

df = pd.read_sql_query(
    "SELECT * FROM futures_data",
    conn
)

conn.close()

df.head()

In [ ]:
import sqlite3
import pandas as pd

db = "../database/futures.db"

conn = sqlite3.connect(db)

df = pd.read_sql_query(
    "SELECT * FROM futures_data",
    conn
)

conn.close()

df.head()

In [ ]:
print("Total rows:", len(df))
print("Total contracts:", df["symbol"].nunique())
print("Total indices:", df["symbol"].str.extract(r"(?:NSE|BSE):([A-Z]+)")[0].nunique())

print("\nDate range:")
print("From:", df["datetime"].min())
print("To  :", df["datetime"].max())

print("\nColumns:")
print(df.columns.tolist())

In [ ]:
# ============================================================
# ANALYSIS TIMEFRAME
# ============================================================

TIMEFRAME = "1D"

# Examples:
# "5min"  → 5-minute
# "15min" → 15-minute
# "30min" → 30-minute
# "1h"    → 1-hour
# "4h"    → 4-hour
# "1D"    → 1-day
# "1W"    → 1-week

print("Selected timeframe:", TIMEFRAME)

In [ ]:
# ============================================================
# RESAMPLE + PRICE / OI / VOLUME CHANGES
# ============================================================

analysis_df = df.copy()

analysis_df["datetime"] = pd.to_datetime(
    analysis_df["datetime"]
)

analysis_df = analysis_df.sort_values(
    ["symbol", "datetime"]
)

analysis_df = (
    analysis_df
    .set_index("datetime")
    .groupby(["symbol", "contract", "expiry"])
    .resample(TIMEFRAME)
    .agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",
            "volume": "sum",
            "open_interest": "last",
        }
    )
    .dropna(subset=["open", "high", "low", "close"])
    .reset_index()
)

# ------------------------------------------------------------
# PREVIOUS VALUES
# ------------------------------------------------------------

analysis_df["previous_close"] = (
    analysis_df
    .groupby("symbol")["close"]
    .shift(1)
)

analysis_df["previous_oi"] = (
    analysis_df
    .groupby("symbol")["open_interest"]
    .shift(1)
)

analysis_df["previous_volume"] = (
    analysis_df
    .groupby("symbol")["volume"]
    .shift(1)
)

# ------------------------------------------------------------
# CHANGE
# ------------------------------------------------------------

analysis_df["ltp_change"] = (
    analysis_df["close"]
    - analysis_df["previous_close"]
)

analysis_df["oi_change"] = (
    analysis_df["open_interest"]
    - analysis_df["previous_oi"]
)

analysis_df["volume_change"] = (
    analysis_df["volume"]
    - analysis_df["previous_volume"]
)

# ------------------------------------------------------------
# CHANGE %
# ------------------------------------------------------------

analysis_df["ltp_change_pct"] = (
    analysis_df["ltp_change"]
    / analysis_df["previous_close"]
    * 100
)

analysis_df["oi_change_pct"] = (
    analysis_df["oi_change"]
    / analysis_df["previous_oi"]
    * 100
)

analysis_df["volume_change_pct"] = (
    analysis_df["volume_change"]
    / analysis_df["previous_volume"]
    * 100
)

print("Timeframe:", TIMEFRAME)
print("Rows:", len(analysis_df))

analysis_df.head(10)

In [ ]:
# ============================================================
# FUTURES ANALYSIS
# FLEXIBLE TIMEFRAME + PRICE/OI/VOLUME CHANGES
# ============================================================

# ------------------------------------------------------------
# SELECT TIMEFRAME
# ------------------------------------------------------------

TIMEFRAME = "1D"

# Examples:
# "5min"   → 5-minute
# "15min"  → 15-minute
# "30min"  → 30-minute
# "1h"     → 1-hour
# "4h"     → 4-hour
# "1D"     → 1-day
# "1W"     → 1-week


# ------------------------------------------------------------
# COPY RAW DATA
# ------------------------------------------------------------

analysis_df = df.copy()


# ------------------------------------------------------------
# DATETIME
# ------------------------------------------------------------

analysis_df["datetime"] = pd.to_datetime(
    analysis_df["datetime"]
)


# ------------------------------------------------------------
# SORT
# ------------------------------------------------------------

analysis_df = analysis_df.sort_values(
    ["symbol", "contract", "datetime"]
)


# ------------------------------------------------------------
# RESAMPLE
# ------------------------------------------------------------

analysis_df = (
    analysis_df
    .set_index("datetime")
    .groupby(
        ["symbol", "contract", "expiry"]
    )
    .resample(TIMEFRAME)
    .agg(
        {
            "open": "first",
            "high": "max",
            "low": "min",
            "close": "last",

            # Total volume during the timeframe
            "volume": "sum",

            # OI at the end of the timeframe
            "open_interest": "last",
        }
    )
    .dropna(
        subset=[
            "open",
            "high",
            "low",
            "close"
        ]
    )
    .reset_index()
)


# ------------------------------------------------------------
# SORT AGAIN
# ------------------------------------------------------------

analysis_df = analysis_df.sort_values(
    ["symbol", "contract", "datetime"]
).reset_index(drop=True)


# ============================================================
# PREVIOUS VALUES
# IMPORTANT:
# COMPARE WITHIN THE SAME CONTRACT ONLY
# ============================================================

group_columns = [
    "symbol",
    "contract"
]


analysis_df["previous_close"] = (
    analysis_df
    .groupby(group_columns)["close"]
    .shift(1)
)


analysis_df["previous_oi"] = (
    analysis_df
    .groupby(group_columns)["open_interest"]
    .shift(1)
)


analysis_df["previous_volume"] = (
    analysis_df
    .groupby(group_columns)["volume"]
    .shift(1)
)


# ============================================================
# ABSOLUTE CHANGES
# ============================================================

analysis_df["ltp_change"] = (
    analysis_df["close"]
    - analysis_df["previous_close"]
)


analysis_df["oi_change"] = (
    analysis_df["open_interest"]
    - analysis_df["previous_oi"]
)


analysis_df["volume_change"] = (
    analysis_df["volume"]
    - analysis_df["previous_volume"]
)


# ============================================================
# PERCENTAGE CHANGES
# ============================================================

# Price / LTP Change %

analysis_df["ltp_change_pct"] = (
    analysis_df["ltp_change"]
    .div(analysis_df["previous_close"])
    .mul(100)
)


# OI Change %

# If previous OI = 0, percentage change is undefined.
analysis_df["oi_change_pct"] = (
    analysis_df["oi_change"]
    .div(
        analysis_df["previous_oi"]
        .replace(0, pd.NA)
    )
    .mul(100)
)


# Volume Change %

# If previous volume = 0, percentage change is undefined.
analysis_df["volume_change_pct"] = (
    analysis_df["volume_change"]
    .div(
        analysis_df["previous_volume"]
        .replace(0, pd.NA)
    )
    .mul(100)
)


# ============================================================
# DISPLAY
# ============================================================

print("=" * 70)
print("FUTURES ANALYSIS DATA")
print("=" * 70)

print("Selected timeframe :", TIMEFRAME)
print("Total rows         :", len(analysis_df))
print(
    "Total contracts    :",
    analysis_df["symbol"].nunique()
)

print()
print("Columns available:")

print(
    analysis_df.columns.tolist()
)


# ============================================================
# SHOW SAMPLE
# ============================================================

analysis_df.head(10)

In [ ]:
# ============================================================
# LONG / SHORT BUILDUP CLASSIFICATION
# ============================================================

def classify_position(row):

    price_change = row["ltp_change"]
    oi_change = row["oi_change"]

    if pd.isna(price_change) or pd.isna(oi_change):
        return "No Data"

    if price_change > 0 and oi_change > 0:
        return "Long Buildup"

    elif price_change < 0 and oi_change > 0:
        return "Short Buildup"

    elif price_change > 0 and oi_change < 0:
        return "Short Covering"

    elif price_change < 0 and oi_change < 0:
        return "Long Unwinding"

    elif price_change == 0 and oi_change > 0:
        return "Price Flat + OI Increase"

    elif price_change == 0 and oi_change < 0:
        return "Price Flat + OI Decrease"

    else:
        return "Neutral"


analysis_df["position"] = analysis_df.apply(
    classify_position,
    axis=1
)


# ============================================================
# DISPLAY IMPORTANT COLUMNS
# ============================================================

display_columns = [
    "symbol",
    "contract",
    "expiry",
    "datetime",
    "close",
    "open_interest",
    "ltp_change_pct",
    "oi_change_pct",
    "volume_change_pct",
    "position"
]

analysis_df[display_columns].head(20)

In [ ]:
# ============================================================
# STUDY SETTINGS
# ============================================================

TIMEFRAME = "1D"

INDEX = "NIFTY"

CONTRACT = "CURRENT"

print("Index    :", INDEX)
print("Contract :", CONTRACT)
print("Timeframe:", TIMEFRAME)

In [ ]:
# ============================================================
# AUTOMATIC CONTRACT SELECTION
# ============================================================

from datetime import datetime
from zoneinfo import ZoneInfo

IST = ZoneInfo("Asia/Kolkata")

today = datetime.now(IST).date()

# ------------------------------------------------------------
# AVAILABLE CONTRACTS FOR SELECTED INDEX
# ------------------------------------------------------------

index_contracts = (
    analysis_df[
        analysis_df["symbol"].str.upper().str.endswith(
            INDEX.upper() + "FUT"
        )
        | (
            analysis_df["contract"]
            .str.upper()
            .str.startswith(INDEX.upper() + " ")
        )
    ][
        ["symbol", "contract", "expiry"]
    ]
    .drop_duplicates()
    .copy()
)

# ------------------------------------------------------------
# CONVERT EXPIRY TO DATE
# ------------------------------------------------------------

index_contracts["expiry_date"] = pd.to_datetime(
    index_contracts["expiry"]
).dt.date

# ------------------------------------------------------------
# REMOVE EXPIRED CONTRACTS
# ------------------------------------------------------------

active_contracts = index_contracts[
    index_contracts["expiry_date"] >= today
].sort_values(
    "expiry_date"
).reset_index(drop=True)

# ------------------------------------------------------------
# SELECT CURRENT / NEXT / FAR
# ------------------------------------------------------------

positions = {
    "CURRENT": 0,
    "NEXT": 1,
    "FAR": 2
}

if CONTRACT not in positions:
    raise ValueError(
        "CONTRACT must be CURRENT, NEXT, or FAR"
    )

position_number = positions[CONTRACT]

if len(active_contracts) <= position_number:
    raise ValueError(
        f"{INDEX} does not have enough active "
        f"contracts in the database for {CONTRACT}."
    )

selected_contract = active_contracts.iloc[position_number]

selected_symbol = selected_contract["symbol"]
selected_contract_name = selected_contract["contract"]
selected_expiry = selected_contract["expiry_date"]

print("=" * 70)
print("SELECTED CONTRACT")
print("=" * 70)

print("Today       :", today)
print("Index       :", INDEX)
print("Position    :", CONTRACT)
print("Timeframe   :", TIMEFRAME)
print("Symbol      :", selected_symbol)
print("Contract    :", selected_contract_name)
print("Expiry      :", selected_expiry)

In [ ]:
# ============================================================
# CUMULATIVE ANALYSIS OF CURRENT + NEXT + FAR
# ============================================================
#
# USE CASE:
# ------------------------------------------------------------
# This section combines the 3 active futures contracts of the
# selected index:
#
#     CURRENT + NEXT + FAR
#
# It helps us understand the TOTAL futures participation of
# the index instead of looking at only one contract.
#
# Useful for:
#
# 1. Total Open Interest
#    → How much futures positioning exists across all 3
#      contracts combined?
#
# 2. Total OI Change
#    → Is overall futures positioning increasing or decreasing?
#
# 3. Total Volume
#    → How much trading activity is happening across the
#      complete futures curve?
#
# 4. Total Volume Change %
#    → Is participation increasing or decreasing?
#
# 5. OI Change %
#    → Overall increase/decrease in futures positioning.
#
# IMPORTANT:
# ------------------------------------------------------------
# LTP cannot simply be added across contracts.
#
# Therefore, we calculate an OI-weighted average LTP.
#
# This gives a representative price of the combined futures
# positions.
# ============================================================


# ------------------------------------------------------------
# GET CURRENT / NEXT / FAR CONTRACTS
# FOR THE SELECTED INDEX
# ------------------------------------------------------------

selected_contracts = (
    active_contracts
    .head(3)
    .copy()
)

selected_symbols = (
    selected_contracts["symbol"]
    .tolist()
)

print("=" * 70)
print("CURRENT + NEXT + FAR — CUMULATIVE ANALYSIS")
print("=" * 70)

print("Index:", INDEX)

print()
print("Contracts included:")

for _, row in selected_contracts.iterrows():

    print(
        f"{row['symbol']} | "
        f"Expiry: {row['expiry_date']}"
    )


# ------------------------------------------------------------
# FILTER ANALYSIS DATA
# ------------------------------------------------------------

cumulative_df = analysis_df[
    analysis_df["symbol"].isin(
        selected_symbols
    )
].copy()


# ------------------------------------------------------------
# OI-WEIGHTED LTP
# ------------------------------------------------------------
#
# LTP is NOT additive.
#
# Example:
# CURRENT LTP = 25,000
# NEXT LTP    = 25,200
# FAR LTP     = 25,500
#
# We cannot do:
#
# 25,000 + 25,200 + 25,500
#
# Instead we calculate an OI-weighted average.
# ------------------------------------------------------------

cumulative_df["weighted_ltp"] = (
    cumulative_df["close"]
    * cumulative_df["open_interest"]
)


# ------------------------------------------------------------
# DAILY CUMULATIVE VALUES
# ------------------------------------------------------------

cumulative_daily = (
    cumulative_df
    .groupby("datetime")
    .agg(
        total_volume=(
            "volume",
            "sum"
        ),

        total_oi=(
            "open_interest",
            "sum"
        ),

        total_oi_change=(
            "oi_change",
            "sum"
        ),

        total_volume_change=(
            "volume_change",
            "sum"
        ),

        weighted_ltp_numerator=(
            "weighted_ltp",
            "sum"
        ),

        total_oi_weight=(
            "open_interest",
            "sum"
        )
    )
    .reset_index()
)


# ------------------------------------------------------------
# COMBINED OI-WEIGHTED LTP
# ------------------------------------------------------------

cumulative_daily["ltp"] = (
    cumulative_daily["weighted_ltp_numerator"]
    / cumulative_daily["total_oi_weight"]
)


# ------------------------------------------------------------
# PREVIOUS CUMULATIVE VALUES
# ------------------------------------------------------------

cumulative_daily = cumulative_daily.sort_values(
    "datetime"
).reset_index(drop=True)


cumulative_daily["previous_total_oi"] = (
    cumulative_daily["total_oi"]
    .shift(1)
)


cumulative_daily["previous_total_volume"] = (
    cumulative_daily["total_volume"]
    .shift(1)
)


cumulative_daily["previous_ltp"] = (
    cumulative_daily["ltp"]
    .shift(1)
)


# ------------------------------------------------------------
# CUMULATIVE OI CHANGE %
# ------------------------------------------------------------

cumulative_daily["total_oi_change_pct"] = (
    cumulative_daily["total_oi_change"]
    .div(
        cumulative_daily["previous_total_oi"]
        .replace(0, pd.NA)
    )
    * 100
)


# ------------------------------------------------------------
# CUMULATIVE VOLUME CHANGE %
# ------------------------------------------------------------

cumulative_daily["total_volume_change_pct"] = (
    (
        cumulative_daily["total_volume"]
        - cumulative_daily["previous_total_volume"]
    )
    .div(
        cumulative_daily["previous_total_volume"]
        .replace(0, pd.NA)
    )
    * 100
)


# ------------------------------------------------------------
# COMBINED LTP CHANGE %
# ------------------------------------------------------------

cumulative_daily["ltp_change_pct"] = (
    (
        cumulative_daily["ltp"]
        - cumulative_daily["previous_ltp"]
    )
    .div(
        cumulative_daily["previous_ltp"]
    )
    * 100
)


# ------------------------------------------------------------
# CLEAN DISPLAY
# ------------------------------------------------------------

cumulative_display = cumulative_daily[
    [
        "datetime",
        "ltp",
        "total_volume",
        "total_oi",
        "total_oi_change",
        "total_oi_change_pct",
        "ltp_change_pct",
        "total_volume_change_pct"
    ]
].copy()


print()
print("=" * 70)
print("CUMULATIVE RESULTS")
print("=" * 70)

cumulative_display.tail(20)

In [ ]:
# ============================================================
# CUMULATIVE BUILDUP / UNWINDING ANALYSIS
# CURRENT + NEXT + FAR
# ============================================================
#
# USE CASE:
# ------------------------------------------------------------
# This classifies the combined futures position of:
#
#     CURRENT + NEXT + FAR
#
# It helps identify whether overall futures participation
# is showing:
#
# 🟢 Long Buildup
# 🔴 Short Buildup
# 🔵 Short Covering
# 🟠 Long Unwinding
#
# The classification is based on:
#
#     Combined LTP Change
#     Combined OI Change
#
# ------------------------------------------------------------


# ============================================================
# CLASSIFICATION FUNCTION
# ============================================================

def classify_cumulative(row):

    ltp_change = row["ltp_change_pct"]
    oi_change = row["total_oi_change_pct"]

    if pd.isna(ltp_change) or pd.isna(oi_change):
        return "⚪ No Data"

    if ltp_change > 0 and oi_change > 0:
        return "🟢 Long Buildup"

    elif ltp_change < 0 and oi_change > 0:
        return "🔴 Short Buildup"

    elif ltp_change > 0 and oi_change < 0:
        return "🔵 Short Covering"

    elif ltp_change < 0 and oi_change < 0:
        return "🟠 Long Unwinding"

    elif ltp_change == 0 and oi_change > 0:
        return "🟡 Price Flat + OI Increase"

    elif ltp_change == 0 and oi_change < 0:
        return "🟣 Price Flat + OI Decrease"

    else:
        return "⚪ Neutral"


# ============================================================
# APPLY CLASSIFICATION
# ============================================================

cumulative_daily["cumulative_position"] = (
    cumulative_daily.apply(
        classify_cumulative,
        axis=1
    )
)


# ============================================================
# CREATE DISPLAY TABLE
# ============================================================
#
# IMPORTANT:
# The original numeric columns remain unchanged.
#
# We create a separate display dataframe so that "%":
# symbols do not turn our calculation columns into text.
# ============================================================

cumulative_display = cumulative_daily[
    [
        "datetime",
        "ltp",
        "total_volume",
        "total_oi",
        "total_oi_change",
        "total_oi_change_pct",
        "ltp_change_pct",
        "total_volume_change_pct",
        "cumulative_position"
    ]
].copy()


# ============================================================
# FORMAT PERCENTAGES
# ============================================================

cumulative_display["total_oi_change_pct"] = (
    cumulative_display["total_oi_change_pct"]
    .apply(
        lambda x:
        f"{x:.2f}%"
        if pd.notna(x)
        else "N/A"
    )
)


cumulative_display["ltp_change_pct"] = (
    cumulative_display["ltp_change_pct"]
    .apply(
        lambda x:
        f"{x:.2f}%"
        if pd.notna(x)
        else "N/A"
    )
)


cumulative_display["total_volume_change_pct"] = (
    cumulative_display["total_volume_change_pct"]
    .apply(
        lambda x:
        f"{x:.2f}%"
        if pd.notna(x)
        else "N/A"
    )
)


# ============================================================
# ROUND NUMERIC VALUES FOR DISPLAY
# ============================================================

cumulative_display["ltp"] = (
    cumulative_display["ltp"]
    .round(2)
)

cumulative_display["total_volume"] = (
    cumulative_display["total_volume"]
    .round(0)
)

cumulative_display["total_oi"] = (
    cumulative_display["total_oi"]
    .round(0)
)

cumulative_display["total_oi_change"] = (
    cumulative_display["total_oi_change"]
    .round(0)
)


# ============================================================
# DISPLAY LATEST RESULTS
# ============================================================

print("=" * 90)
print("CUMULATIVE FUTURES POSITION")
print("=" * 90)

print(
    f"Index: {INDEX} | "
    f"Timeframe: {TIMEFRAME}"
)

print(
    "Contracts: CURRENT + NEXT + FAR"
)

print()

cumulative_display.tail(20)

In [ ]:
# ============================================================
# STATISTICAL ANALYSIS
# ============================================================
#
# USE CASE:
# ------------------------------------------------------------
# This section measures price movement and volatility.
#
# Standard deviation is calculated on LTP returns rather than
# the absolute LTP.
#
# Rolling standard deviation helps us identify periods where
# price movement is unusually calm or volatile.
#
# The window automatically follows the selected timeframe.
#
# Examples:
#
# TIMEFRAME = "1D"  → 20 periods = approximately 20 trading days
# TIMEFRAME = "1h"  → 20 periods = approximately 20 hourly candles
# TIMEFRAME = "5min" → 20 periods = 20 five-minute candles
# ============================================================


# ------------------------------------------------------------
# SELECT CONTRACT DATA
# ------------------------------------------------------------

selected_df = analysis_df[
    analysis_df["symbol"] == selected_symbol
].copy()


selected_df = selected_df.sort_values(
    "datetime"
).reset_index(drop=True)


# ------------------------------------------------------------
# PRICE RETURN
# ------------------------------------------------------------

selected_df["return_pct"] = (
    selected_df["close"]
    .pct_change()
    * 100
)


# ------------------------------------------------------------
# ROLLING WINDOWS
# ------------------------------------------------------------

ROLLING_SHORT = 5
ROLLING_MEDIUM = 10
ROLLING_LONG = 20


# ------------------------------------------------------------
# ROLLING STANDARD DEVIATION
# ------------------------------------------------------------

selected_df["std_5"] = (
    selected_df["return_pct"]
    .rolling(ROLLING_SHORT)
    .std()
)


selected_df["std_10"] = (
    selected_df["return_pct"]
    .rolling(ROLLING_MEDIUM)
    .std()
)


selected_df["std_20"] = (
    selected_df["return_pct"]
    .rolling(ROLLING_LONG)
    .std()
)


# ------------------------------------------------------------
# OI STANDARD DEVIATION
# ------------------------------------------------------------

selected_df["oi_std_20"] = (
    selected_df["open_interest"]
    .rolling(ROLLING_LONG)
    .std()
)


# ------------------------------------------------------------
# VOLUME STANDARD DEVIATION
# ------------------------------------------------------------

selected_df["volume_std_20"] = (
    selected_df["volume"]
    .rolling(ROLLING_LONG)
    .std()
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("=" * 80)
print("STATISTICAL ANALYSIS")
print("=" * 80)

print("Index     :", INDEX)
print("Contract  :", CONTRACT)
print("Symbol    :", selected_symbol)
print("Timeframe :", TIMEFRAME)

print()
print("Return statistics:")
print(
    selected_df["return_pct"]
    .describe()
)


# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

statistics_display = selected_df[
    [
        "datetime",
        "close",
        "open_interest",
        "volume",
        "return_pct",
        "std_5",
        "std_10",
        "std_20",
        "oi_std_20",
        "volume_std_20"
    ]
].copy()


statistics_display.tail(20)

In [ ]:
# ============================================================
# CHECK NIFTY CONTRACT REPRESENTATIONS
# ============================================================

nifty_check = df[
    df["symbol"] == "NSE:NIFTY26SEPFUT"
].copy()

print("NIFTY CURRENT contract values:")
print(
    nifty_check["contract"]
    .value_counts()
)

print()
print("Total NIFTY CURRENT rows:", len(nifty_check))

In [ ]:
# ============================================================
# CLEAN ANALYSIS INPUT
# ============================================================

analysis_input = df.copy()

# ------------------------------------------------------------
# DATETIME
# ------------------------------------------------------------

analysis_input["datetime"] = pd.to_datetime(
    analysis_input["datetime"]
)

In [ ]:
# ============================================================
# CHECK DAILY DATA
# ============================================================

print("Timeframe:", TIMEFRAME)
print("Total analysis rows:", len(analysis_df))

print()
print("Duplicate symbol + contract + date:")
print(
    analysis_df
    .assign(date=analysis_df["datetime"].dt.date)
    .duplicated(
        subset=["symbol", "contract", "date"]
    )
    .sum()
)

print()
print("NIFTY CURRENT daily observations:")

display(
    analysis_df[
        analysis_df["symbol"] == selected_symbol
    ][
        [
            "datetime",
            "close",
            "open_interest",
            "volume"
        ]
    ].tail(15)
)

In [ ]:
# ============================================================
# PROFESSIONAL FUTURES TRADING DASHBOARD
# ============================================================
#
# FEATURES
# ------------------------------------------------------------
# • Green / Red candlesticks
# • No weekend gaps
# • No holiday gaps
# • Continuous trading-session X-axis
# • Actual trading dates shown below candles
# • Timeframe shown below dates
# • EMA 9 / EMA 20 / EMA 50
# • Volume
# • Open Interest
# • OI Change %
# • Price Return %
# • Rolling Standard Deviation
# • Automatically works with:
#       5min
#       15min
#       30min
#       1h
#       4h
#       1D
#       1W
#
# RAW DATABASE IS NOT MODIFIED.
# ============================================================


import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import pandas as pd
import numpy as np


# ============================================================
# 1. SELECTED CONTRACT
# ============================================================

plot_df = analysis_df[
    (analysis_df["symbol"] == selected_symbol) &
    (analysis_df["contract"] == selected_contract_name)
].copy()


# ============================================================
# 2. CHECK DATA
# ============================================================

if plot_df.empty:

    raise ValueError(
        "No analysis data found for the selected contract."
    )


# ============================================================
# 3. DATETIME
# ============================================================

plot_df["datetime"] = pd.to_datetime(
    plot_df["datetime"]
)


plot_df = (
    plot_df
    .sort_values("datetime")
    .reset_index(drop=True)
)


# ============================================================
# 4. REMOVE WEEKENDS
# ============================================================
#
# This affects ONLY the chart.
#
# Saturday = 5
# Sunday   = 6
#
# Holidays are naturally removed because there are no
# trading candles for those dates.
#
# ============================================================

plot_df = plot_df[
    plot_df["datetime"].dt.dayofweek < 5
].copy()


plot_df = (
    plot_df
    .sort_values("datetime")
    .reset_index(drop=True)
)


# ============================================================
# 5. CHECK DATA AFTER FILTERING
# ============================================================

if plot_df.empty:

    raise ValueError(
        "No weekday trading data available for the selected contract."
    )


# ============================================================
# 6. CREATE CONTINUOUS TRADING CANDLE INDEX
# ============================================================
#
# This is the important part.
#
# Instead of:
#
#       01-Jul
#       02-Jul
#       03-Jul
#       04-Jul
#       05-Jul
#       06-Jul
#
# the chart internally uses:
#
#       0
#       1
#       2
#       3
#       4
#       5
#
# Therefore weekends and holidays create NO visual gaps.
#
# ============================================================

plot_df["candle_index"] = np.arange(
    len(plot_df)
)


x = plot_df["candle_index"].to_numpy()


# ============================================================
# 7. INDICATORS
# ============================================================


# ------------------------------------------------------------
# EMA 9
# ------------------------------------------------------------

plot_df["EMA_9"] = (
    plot_df["close"]
    .ewm(
        span=9,
        adjust=False
    )
    .mean()
)


# ------------------------------------------------------------
# EMA 20
# ------------------------------------------------------------

plot_df["EMA_20"] = (
    plot_df["close"]
    .ewm(
        span=20,
        adjust=False
    )
    .mean()
)


# ------------------------------------------------------------
# EMA 50
# ------------------------------------------------------------

plot_df["EMA_50"] = (
    plot_df["close"]
    .ewm(
        span=50,
        adjust=False
    )
    .mean()
)


# ------------------------------------------------------------
# PRICE RETURN %
# ------------------------------------------------------------

plot_df["return_pct"] = (
    plot_df["close"]
    .pct_change()
    * 100
)


# ------------------------------------------------------------
# OI CHANGE %
# ------------------------------------------------------------

plot_df["oi_change_pct"] = (
    plot_df["open_interest"]
    .pct_change()
    * 100
)


# ------------------------------------------------------------
# ROLLING STANDARD DEVIATION
# ------------------------------------------------------------

plot_df["std_5"] = (
    plot_df["return_pct"]
    .rolling(5)
    .std()
)


plot_df["std_20"] = (
    plot_df["return_pct"]
    .rolling(20)
    .std()
)


# ============================================================
# 8. TIMEFRAME DISPLAY NAME
# ============================================================

timeframe_names = {

    "5min": "5 Minute",

    "15min": "15 Minute",

    "30min": "30 Minute",

    "1h": "1 Hour",

    "4h": "4 Hour",

    "1D": "Daily",

    "1W": "Weekly"
}


timeframe_text = timeframe_names.get(
    TIMEFRAME,
    TIMEFRAME
)


# ============================================================
# 9. CANDLE WIDTH
# ============================================================
#
# Because the X-axis is now a candle index, one candle
# always occupies approximately one unit.
#
# ============================================================

candle_width = 0.65


# ============================================================
# 10. CREATE FIGURE
# ============================================================

fig = plt.figure(
    figsize=(18, 22)
)


# ============================================================
# 11. GRID LAYOUT
# ============================================================

gs = fig.add_gridspec(

    6,
    1,

    height_ratios=[
        5.0,   # Price
        1.4,   # Volume
        2.0,   # OI
        1.5,   # OI Change
        1.5,   # Return
        1.5    # Standard deviation
    ],

    hspace=0.24
)


# ============================================================
# 12. CREATE AXES
# ============================================================

ax_price = fig.add_subplot(
    gs[0]
)


ax_volume = fig.add_subplot(
    gs[1]
)


ax_oi = fig.add_subplot(
    gs[2]
)


ax_oi_change = fig.add_subplot(
    gs[3]
)


ax_return = fig.add_subplot(
    gs[4]
)


ax_std = fig.add_subplot(
    gs[5]
)


# ============================================================
# 13. X-AXIS RANGE
# ============================================================

xmin = -1

xmax = len(plot_df)


for ax in [

    ax_price,

    ax_volume,

    ax_oi,

    ax_oi_change,

    ax_return,

    ax_std

]:

    ax.set_xlim(
        xmin,
        xmax
    )


# ============================================================
# 14. CANDLESTICK CHART
# ============================================================

for i, row in enumerate(
    plot_df.itertuples()
):

    candle_x = i


    open_price = row.open

    high_price = row.high

    low_price = row.low

    close_price = row.close


    # --------------------------------------------------------
    # CANDLE COLOR
    # --------------------------------------------------------
    #
    # GREEN = Close >= Open
    # RED   = Close < Open
    #
    # --------------------------------------------------------

    if close_price >= open_price:

        candle_color = "green"

    else:

        candle_color = "red"


    # --------------------------------------------------------
    # WICK
    # --------------------------------------------------------

    ax_price.plot(

        [candle_x, candle_x],

        [low_price, high_price],

        color=candle_color,

        linewidth=1.2,

        solid_capstyle="round"
    )


    # --------------------------------------------------------
    # BODY
    # --------------------------------------------------------

    body_low = min(
        open_price,
        close_price
    )


    body_height = abs(
        close_price - open_price
    )


    # --------------------------------------------------------
    # DOJI
    # --------------------------------------------------------

    if body_height == 0:

        body_height = max(

            (high_price - low_price)
            * 0.02,

            0.5
        )


    # --------------------------------------------------------
    # DRAW CANDLE
    # --------------------------------------------------------

    candle = Rectangle(

        (
            candle_x - candle_width / 2,

            body_low
        ),

        candle_width,

        body_height,

        facecolor=candle_color,

        edgecolor=candle_color,

        linewidth=0.8,

        alpha=0.90
    )


    ax_price.add_patch(
        candle
    )


# ============================================================
# 15. MOVING AVERAGES
# ============================================================

ax_price.plot(

    x,

    plot_df["EMA_9"],

    linewidth=1.5,

    label="EMA 9"
)


ax_price.plot(

    x,

    plot_df["EMA_20"],

    linewidth=1.5,

    label="EMA 20"
)


ax_price.plot(

    x,

    plot_df["EMA_50"],

    linewidth=1.5,

    label="EMA 50"
)


# ============================================================
# 16. CURRENT PRICE
# ============================================================

latest_price = (
    plot_df["close"].iloc[-1]
)


latest_x = (
    plot_df["candle_index"].iloc[-1]
)


# Current price horizontal line

ax_price.axhline(

    latest_price,

    linestyle="--",

    linewidth=1
)


# Current price label

ax_price.annotate(

    f"{latest_price:,.2f}",

    xy=(

        latest_x,

        latest_price
    ),

    xytext=(

        8,

        0
    ),

    textcoords="offset points",

    va="center",

    fontsize=9,

    fontweight="bold"
)


# ============================================================
# 17. PRICE PANEL TITLE
# ============================================================

ax_price.set_title(

    f"{INDEX} FUTURES — {CONTRACT}\n"
    f"{selected_contract_name}",

    fontsize=16,

    fontweight="bold",

    pad=10
)


ax_price.set_ylabel(

    "Price",

    fontsize=10,

    fontweight="bold"
)


ax_price.legend(

    loc="upper left",

    frameon=True
)


ax_price.grid(

    True,

    alpha=0.20,

    linestyle="--"
)


# ============================================================
# 18. TRADING DATE LABELS
# ============================================================
#
# We show actual dates from the database.
#
# The positions are based on candle_index, NOT calendar dates.
#
# Therefore:
#
# Friday → Monday
#
# has no empty weekend gap.
#
# ============================================================

max_labels = 10


if len(plot_df) <= max_labels:

    label_positions = np.arange(
        len(plot_df)
    )

else:

    label_positions = np.linspace(

        0,

        len(plot_df) - 1,

        max_labels,

        dtype=int
    )


label_positions = np.unique(
    label_positions
)


label_text = []


for position in label_positions:

    dt = plot_df.loc[
        position,
        "datetime"
    ]


    # --------------------------------------------------------
    # Intraday timeframe
    # --------------------------------------------------------

    if TIMEFRAME in [

        "5min",

        "15min",

        "30min",

        "1h",

        "4h"

    ]:

        label_text.append(

            dt.strftime(
                "%d-%b\n%H:%M"
            )
        )


    # --------------------------------------------------------
    # Daily / Weekly
    # --------------------------------------------------------

    else:

        label_text.append(

            dt.strftime(
                "%d-%b"
            )
        )


# ============================================================
# 19. APPLY DATE LABELS TO PRICE PANEL
# ============================================================

ax_price.set_xticks(
    label_positions
)


ax_price.set_xticklabels(
    label_text
)


ax_price.tick_params(

    axis="x",

    which="both",

    bottom=True,

    labelbottom=True,

    labelsize=9,

    pad=5
)


# ============================================================
# 20. TIMEFRAME LABEL
# ============================================================

ax_price.set_xlabel(

    f"DATE / TIME\n"
    f"TIMEFRAME: {TIMEFRAME} ({timeframe_text})",

    fontsize=10,

    fontweight="bold",

    labelpad=10
)


# ============================================================
# 21. VOLUME
# ============================================================

volume_colors = np.where(

    plot_df["close"]
    >= plot_df["open"],

    "green",

    "red"
)


ax_volume.bar(

    x,

    plot_df["volume"],

    width=candle_width,

    color=volume_colors,

    alpha=0.65
)


ax_volume.set_ylabel(

    "Volume",

    fontweight="bold"
)


ax_volume.set_title(

    "Volume",

    loc="left",

    fontsize=11,

    fontweight="bold"
)


ax_volume.grid(

    True,

    alpha=0.20,

    linestyle="--"
)


# ============================================================
# 22. OPEN INTEREST
# ============================================================

ax_oi.plot(

    x,

    plot_df["open_interest"],

    linewidth=1.8
)


# OI area

ax_oi.fill_between(

    x,

    plot_df["open_interest"],

    alpha=0.10
)


ax_oi.set_ylabel(

    "OI",

    fontweight="bold"
)


ax_oi.set_title(

    "Open Interest",

    loc="left",

    fontsize=11,

    fontweight="bold"
)


ax_oi.grid(

    True,

    alpha=0.20,

    linestyle="--"
)


# ============================================================
# 23. OPEN INTEREST CHANGE %
# ============================================================

ax_oi_change.axhline(

    0,

    linewidth=1,

    linestyle="--"
)


oi_colors = np.where(

    plot_df["oi_change_pct"] >= 0,

    "green",

    "red"
)


ax_oi_change.bar(

    x,

    plot_df["oi_change_pct"],

    width=candle_width,

    color=oi_colors,

    alpha=0.65
)


ax_oi_change.set_ylabel(

    "OI %",

    fontweight="bold"
)


ax_oi_change.set_title(

    "Open Interest Change %",

    loc="left",

    fontsize=11,

    fontweight="bold"
)


ax_oi_change.grid(

    True,

    alpha=0.20,

    linestyle="--"
)


# ============================================================
# 24. PRICE RETURN %
# ============================================================

ax_return.axhline(

    0,

    linewidth=1,

    linestyle="--"
)


return_colors = np.where(

    plot_df["return_pct"] >= 0,

    "green",

    "red"
)


ax_return.bar(

    x,

    plot_df["return_pct"],

    width=candle_width,

    color=return_colors,

    alpha=0.65
)


ax_return.set_ylabel(

    "Return %",

    fontweight="bold"
)


ax_return.set_title(

    "Price Return %",

    loc="left",

    fontsize=11,

    fontweight="bold"
)


ax_return.grid(

    True,

    alpha=0.20,

    linestyle="--"
)


# ============================================================
# 25. STANDARD DEVIATION
# ============================================================

ax_std.plot(

    x,

    plot_df["std_5"],

    linewidth=1.6,

    label="5 Period"
)


ax_std.plot(

    x,

    plot_df["std_20"],

    linewidth=1.6,

    label="20 Period"
)


ax_std.set_ylabel(

    "Std %",

    fontweight="bold"
)


ax_std.set_title(

    "Rolling Volatility / Standard Deviation",

    loc="left",

    fontsize=11,

    fontweight="bold"
)


ax_std.legend(

    loc="upper left"
)


ax_std.grid(

    True,

    alpha=0.20,

    linestyle="--"
)


# ============================================================
# 26. REMOVE DATE LABELS FROM LOWER PANELS
# ============================================================

for ax in [

    ax_volume,

    ax_oi,

    ax_oi_change,

    ax_return,

    ax_std

]:

    ax.tick_params(

        axis="x",

        labelbottom=False
    )


# ============================================================
# 27. CLEAN SPINES
# ============================================================

for ax in [

    ax_price,

    ax_volume,

    ax_oi,

    ax_oi_change,

    ax_return,

    ax_std

]:

    ax.spines[
        "top"
    ].set_visible(False)


    ax.spines[
        "right"
    ].set_visible(False)


# ============================================================
# 28. SINGLE MAIN TITLE
# ============================================================

fig.suptitle(

    f"{INDEX} FUTURES — {CONTRACT}\n"
    f"{selected_contract_name} | "
    f"Timeframe: {TIMEFRAME} ({timeframe_text})",

    fontsize=18,

    fontweight="bold",

    y=0.995
)


# ============================================================
# 29. FINAL LAYOUT
# ============================================================
#
# Using subplots_adjust instead of tight_layout avoids the
# warning caused by candlestick Rectangle objects.
#
# ============================================================

fig.subplots_adjust(

    left=0.08,

    right=0.97,

    top=0.94,

    bottom=0.06,

    hspace=0.24
)


# ============================================================
# 30. DISPLAY
# ============================================================

plt.show()